In [0]:
%pip install azure-eventhub

In [0]:
%restart_python

In [0]:
# Import the required libraries
from azure.eventhub import EventHubProducerClient, EventData
from datetime import datetime, timezone
import json
import random
import time
import uuid

# Set the Azure Event Hubs namespace and event hub name
EH_NAMESPACE = "hns-orders-demo"
EH_NAME = "orders"

# Retrieve the Event Hubs Send access key securely from Databricks Secrets
# strip("'\"") removes any extra single or double quotes around the key
SEND_KEY = dbutils.secrets.get(
    scope="eh-orders-demo",
    key="send-orders"
).strip("'\"")

# Build the Event Hubs connection string using the namespace and Send key
connection_string = (
    f"Endpoint=sb://{EH_NAMESPACE}.servicebus.windows.net/;"
    f"SharedAccessKeyName=send-orders;"
    f"SharedAccessKey={SEND_KEY}"
)

# Create a producer client to send events to Azure Event Hubs
producer = EventHubProducerClient.from_connection_string(
    conn_str=connection_string,
    eventhub_name=EH_NAME
)

try:
    # Generate and send 20 sample order events
    for i in range(20):

        # Create a sample order event with randomly generated values
        event = {
            # Generate a unique ID for each event
            "event_id": str(uuid.uuid4()),

            # Generate a unique-looking order ID
            "order_id": f"ORD-{uuid.uuid4().hex[:8]}",

            # Select a random customer ID
            "customer_id": f"CUST-{random.randint(1000, 1100)}",

            # Select a random product ID
            "product_id": f"PROD-{random.randint(100, 140)}",

            # Generate an order quantity between 1 and 5
            "quantity": random.randint(1, 5),

            # Generate a random order amount between 100 and 5000
            "amount": round(random.uniform(100, 5000), 2),

            # Randomly select an order status
            "status": random.choice(
                ["PLACED", "PAID", "SHIPPED"]
            ),

            # Record the event creation time in UTC using ISO format
            "event_time": datetime.now(
                timezone.utc
            ).isoformat()
        }

        # Create a batch to hold the event before sending it
        batch = producer.create_batch()

        # Convert the Python dictionary to JSON and add it to the batch
        batch.add(EventData(json.dumps(event)))

        # Send the batch to Azure Event Hubs
        producer.send_batch(batch)

        # Print the order ID to confirm that the send call completed
        print("Sent:", event["order_id"])

        # Wait 3 seconds before generating the next event
        time.sleep(3)

finally:
    # Close the producer connection, even if an error occurs
    producer.close()